<a href="https://colab.research.google.com/github/AlfredaDhaifullah04/PCVK_Ganjil_2026/blob/main/PCVK_M3_W3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modul 3 -  Operasi Dasar Pengolahan Citra


**Mata Kuliah:** Pengolahan Citra dan Visi Komputer  
**Nama:** [Alfreda Dhaifullah Mahezwara]  
**NIM:** [2441107020219]


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import cv2 as cv
import numpy as np
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt
import glob

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Sesuaikan dengan lokasi folder image Anda di Drive
base_path = '/content/drive/MyDrive/Images/'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 1. Inverse Citra
Implementasi inverse citra (citra negatif) dengan rumus: $g(x) = 255 - f(x)$


In [4]:
print('--- Inverse Citra ---')
img_path = base_path + 'peppers.jpg' # Sesuaikan nama file
img = cv.imread(img_path)

if img is not None:
    img_rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    inv_img = 255 - img_rgb

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.title('Citra Asli')
    plt.imshow(img_rgb)

    plt.subplot(1, 2, 2)
    plt.title('Inverse Citra')
    plt.imshow(inv_img)
    plt.show()
else:
    print(f'Gambar tidak ditemukan di {img_path}')


--- Inverse Citra ---
Gambar tidak ditemukan di /content/drive/MyDrive/Images/peppers.jpg


### 2. Transformasi Contrast
Implementasi transformasi kontras dengan rumus: $g(x,y) = a * f(x,y) + b$


In [5]:
print('--- Transformasi Contrast ---')
if img is not None:
    try:
        brightness = int(input('Masukkan tingkat kecerahan (misal: 10): '))
        contrast = float(input('Masukkan kontras (misal: 1.5): '))

        # g(x,y) = a*f(x,y) + b
        # np.clip untuk truncate nilai agar tetap dalam range 0-255
        res = np.clip(contrast * img_rgb + brightness, 0, 255).astype(np.uint8)

        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.title('Citra Asli')
        plt.imshow(img_rgb)

        plt.subplot(1, 2, 2)
        plt.title(f'Contrast: {contrast}, Brightness: {brightness}')
        plt.imshow(res)
        plt.show()
    except ValueError:
        print('Error, not a valid number')


--- Transformasi Contrast ---


### 3. Transformasi Logarithmic Brightness
Formula: $s = c * \log(1+r)$


In [6]:
print('--- Logarithmic Brightness ---')
if img is not None:
    try:
        brightness_log = int(input('Masukkan nilai kecerahan (misal: 50): '))

        c = 255 / np.log(1 + np.max(img_rgb))
        log_img = c * (np.log(img_rgb.astype(np.float32) + 1))

        # Menambahkan nilai kecerahan ekstra
        log_img = np.clip(log_img + brightness_log, 0, 255).astype(np.uint8)

        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.title('Citra Asli')
        plt.imshow(img_rgb)

        plt.subplot(1, 2, 2)
        plt.title('Logarithmic Brightness')
        plt.imshow(log_img)
        plt.show()
    except ValueError:
        print('Error, not a number')


--- Logarithmic Brightness ---


### 4. Grayscale (Averaging, Lightness, Luminance)


In [7]:
print('--- Metode Grayscale ---')
if img is not None:
    # Averaging
    avg_gray = np.mean(img_rgb, axis=2).astype(np.uint8)

    # Lightness
    lightness_gray = ((np.max(img_rgb, axis=2).astype(np.float32) + np.min(img_rgb, axis=2).astype(np.float32)) / 2).astype(np.uint8)

    # Luminance
    # Hati-hati urutan channel, jika dari BGR ke RGB (img_rgb), channelnya adalah R=0, G=1, B=2
    luminance_gray = (0.21 * img_rgb[:,:,0] + 0.72 * img_rgb[:,:,1] + 0.07 * img_rgb[:,:,2]).astype(np.uint8)

    plt.figure(figsize=(15, 5))
    plt.subplot(1, 4, 1), plt.imshow(img_rgb), plt.title('Citra Asli')
    plt.subplot(1, 4, 2), plt.imshow(avg_gray, cmap='gray'), plt.title('Averaging')
    plt.subplot(1, 4, 3), plt.imshow(lightness_gray, cmap='gray'), plt.title('Lightness')
    plt.subplot(1, 4, 4), plt.imshow(luminance_gray, cmap='gray'), plt.title('Luminance')
    plt.show()


--- Metode Grayscale ---


### 5. Filter Warna Tertentu dan Grayscale Background
Menampilkan warna merah saja, sisanya diubah menjadi grayscale.


In [8]:
if img is not None:
    # Konversi ke HSV untuk masking warna yang lebih baik
    hsv = cv.cvtColor(img, cv.COLOR_BGR2HSV)

    # Rentang warna merah di HSV
    lower_red1 = np.array([0, 50, 50])
    upper_red1 = np.array([10, 255, 255])
    mask1 = cv.inRange(hsv, lower_red1, upper_red1)

    lower_red2 = np.array([170, 50, 50])
    upper_red2 = np.array([180, 255, 255])
    mask2 = cv.inRange(hsv, lower_red2, upper_red2)

    mask = mask1 + mask2

    # Buat background grayscale tapi dalam format 3 channel RGB
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    gray_3c = cv.cvtColor(gray, cv.COLOR_GRAY2RGB)

    # Gabungkan
    res_color = np.where(mask[:, :, None] > 0, img_rgb, gray_3c)

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1), plt.imshow(img_rgb), plt.title('Citra Asli')
    plt.subplot(1, 2, 2), plt.imshow(res_color), plt.title('Keep Red, Else Grayscale')
    plt.show()


### 6. Gamma Correction
Formula: $I' = 255 * (I/255)^\gamma$


In [9]:
print('--- Gamma Correction ---')
if img is not None:
    try:
        gamma = float(input('Masukkan nilai Gamma (misal 0.5 atau 3.0): '))
        # Invers Gamma untuk visualisasi karena fungsi pangkat bekerja secara terbalik dengan kecerahan mata
        invGamma = 1.0 / gamma if gamma != 0 else 1.0

        table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype('uint8')
        gamma_img = cv.LUT(img_rgb, table)

        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1), plt.imshow(img_rgb), plt.title('Citra Asli')
        plt.subplot(1, 2, 2), plt.imshow(gamma_img), plt.title(f'Gamma Correction ($\gamma={gamma}$)')
        plt.show()
    except ValueError:
        print('Error, not a number')


--- Gamma Correction ---


<>:13: SyntaxWarning: invalid escape sequence '\g'
<>:13: SyntaxWarning: invalid escape sequence '\g'
/tmp/ipykernel_1074/1742914262.py:13: SyntaxWarning: invalid escape sequence '\g'
  plt.subplot(1, 2, 2), plt.imshow(gamma_img), plt.title(f'Gamma Correction ($\gamma={gamma}$)')


### 7. Simulasi Image (Bit) Depth


In [10]:
print('--- Simulasi Bit Depth ---')
gray_img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
if gray_img is not None:
    try:
        bit_depth = int(input('Masukkan bit depth tujuan (1-8): '))
        if 1 <= bit_depth <= 8:
            level = 255.0 / (2**bit_depth - 1)
            depth_img = np.round(gray_img / level) * level
            depth_img = depth_img.astype(np.uint8)

            print(f'Bit depth awal : 8 bit')
            print(f'Bit depth tujuan: {bit_depth} bit')
            print(f'Jumlah level    : {2**bit_depth}')

            plt.figure(figsize=(10, 5))
            plt.subplot(1, 2, 1), plt.imshow(gray_img, cmap='gray'), plt.title('Grayscale 8-bit')
            plt.subplot(1, 2, 2), plt.imshow(depth_img, cmap='gray'), plt.title(f'Grayscale {bit_depth}-bit')
            plt.show()
        else:
            print('Bit depth harus antara 1 sampai 8')
    except ValueError:
        print('Error, masukkan angka integer')


--- Simulasi Bit Depth ---


### 8. Average Denoising & PSNR
Pastikan Anda memiliki citra galaxy asli (`galaxy.jpg`) dan noise (`noises/*.jpg`) di direktori Drive Anda.


In [11]:
def PSNR(img1, img2):
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100
    max_pixel = 255.0
    psnr = 20 * np.log10(max_pixel / np.sqrt(mse))
    return psnr

# --- JALANKAN INI JIKA ANDA MEMILIKI FILE GALAXY ---
galaxy_original = cv.imread(base_path + 'galaxy.jpg')

cv_img = []
# Membaca seluruh citra bernoise (Pastikan path sesuai)
for img_file in glob.glob(base_path + 'noises/*.jpg'):
    n = cv.imread(img_file)
    if n is not None:
        cv_img.append(n)

if galaxy_original is not None and len(cv_img) > 0:
    # Uji coba untuk 10, 20, 40, 80, 100 citra
    test_cases = [10, 20, 40, 80, 100]

    for jumlah in test_cases:
        if jumlah <= len(cv_img):
            avg_img = np.zeros_like(cv_img[0], dtype=np.float32)
            for i in range(jumlah):
                avg_img += cv_img[i].astype(np.float32)
            avg_img = (avg_img / jumlah).astype(np.uint8)

            nilai_psnr = PSNR(galaxy_original, avg_img)
            print(f'PSNR untuk Average {jumlah} citra: {nilai_psnr:.4f} dB')
else:
    print('Citra galaxy.jpg atau folder noises tidak ditemukan/kosong. Lewati percobaan ini.')


Citra galaxy.jpg atau folder noises tidak ditemukan/kosong. Lewati percobaan ini.


**Kesimpulan Average Denoising:**
1. Peningkatan jumlah citra yang dirata-rata (average) akan meningkatkan nilai PSNR, yang menandakan *noise* semakin berkurang dan citra hasil semakin mendekati citra ideal.
2. Peningkatan signifikan biasanya terjadi di awal (misal dari 10 ke 40 citra). Ketika jumlah citra sudah sangat banyak (misal setelah 80-100), peningkatan PSNR mulai landai (tidak terlalu signifikan) karena *noise* mayoritas sudah saling menghilangkan.
3. **Kesimpulan:** Average denoising sangat efektif untuk mengurangi *Gaussian Noise* acak eksternal dengan cara menumpuk dan merata-rata intensitas pixel, sehingga nilai noise acak saling menetralkan ke nilai tengah (rata-rata).


### 9. Image Masking (Operasi Logika)
Percobaan dengan Operator Logika (NOT, OR, AND, NAND, XOR).


In [12]:
couple_img = cv.imread(base_path + 'couple.tiff')
if couple_img is not None:
    couple_rgb = cv.cvtColor(couple_img, cv.COLOR_BGR2RGB)

    # Membuat Mask berupa dua lingkaran putih
    mask = np.zeros(couple_img.shape[:2], dtype=np.uint8)
    cv.circle(mask, (80, 80), 40, 255, -1) # Kiri
    cv.circle(mask, (180, 80), 40, 255, -1) # Kanan (sesuaikan koordinat)

    mask_3c = cv.cvtColor(mask, cv.COLOR_GRAY2RGB)

    # 1. NOT (Komplemen)
    not_img = cv.bitwise_not(couple_rgb)

    # 2. OR
    or_img = cv.bitwise_or(couple_rgb, mask_3c)

    # 3. AND
    and_img = cv.bitwise_and(couple_rgb, mask_3c)

    # 4. NAND (NOT AND)
    nand_img = cv.bitwise_not(and_img)

    # 5. XOR
    xor_img = cv.bitwise_xor(couple_rgb, mask_3c)

    plt.figure(figsize=(15, 10))
    plt.subplot(2, 3, 1), plt.imshow(couple_rgb), plt.title('Original')
    plt.subplot(2, 3, 2), plt.imshow(mask_3c), plt.title('Mask')
    plt.subplot(2, 3, 3), plt.imshow(not_img), plt.title('NOT Input')
    plt.subplot(2, 3, 4), plt.imshow(or_img), plt.title('OR (Input OR Mask)')
    plt.subplot(2, 3, 5), plt.imshow(and_img), plt.title('AND (Input AND Mask)')
    plt.subplot(2, 3, 6), plt.imshow(xor_img), plt.title('XOR')
    plt.show()


**Analisa Image Masking:**
- **NOT**: Membalikkan seluruh warna pada citra menjadi nilai komplemennya (menghasilkan citra negatif).
- **OR**: Jika diregion mask bernilai 255 (putih), hasilnya pasti putih. Sisanya akan menampilkan citra asli karena 0 OR X = X.
- **AND**: Bagian citra asli hanya akan terlihat pada region mask yang bernilai 255 (putih), karena 255 AND X = X. Bagian hitam (0) akan menutup citra asli karena 0 AND X = 0. Inilah yang biasa disebut **Masking** utama.
- **NAND**: Adalah kebalikan dari hasil AND. Semua yang ditampilkan oleh AND akan di-invers warnanya, dan area yang terpotong menjadi dominan terang.
- **XOR**: Mengeksklusifkan area. Pada area putih mask, warna citra di-invers (kebalikan). Pada area hitam mask, warna citra asli dipertahankan.


### 10. Foto Malam Hari
**Analisis Metode:**
Jika kita memiliki foto malam hari yang sangat gelap namun detailnya ada, metode terbaik yang dapat digunakan adalah **Gamma Correction** (dengan $\gamma < 1$, misalnya $0.4$ atau $0.5$) atau **Logarithmic Brightness**.

**Alasan Pemilihan Gamma Correction / Logarithmic Brightness:**
Metode ini sangat baik dalam merentangkan nilai rentang keabuan (gray-level) yang gelap menjadi lebih lebar, namun tetap mempertahankan detail area terang agar tidak oversaturated. Jika kita menggunakan *Linear Brightness*, semua piksel akan ditambahkan nilai konstan, yang mengakibatkan detail di area terang akan menjadi terlalu putih (wash out/error > 255 ter-truncate).

**Risiko Gamma / Log Brightness:**
Risiko terbesarnya adalah jika foto aslinya memiliki noise sensor kamera yang tinggi (biasa terjadi di ISO tinggi saat malam hari), noise tersebut juga akan ikut tercerahkan dan menjadi lebih terlihat *grainy*.


### 11. Perbaikan Citra `crayfish.jpg`


In [13]:
crayfish = cv.imread(base_path + 'crayfish.jpg')
if crayfish is not None:
    crayfish_rgb = cv.cvtColor(crayfish, cv.COLOR_BGR2RGB)

    # Metode: Kombinasi Transformasi Kontras (Stretching) dan Gamma Correction
    # Langkah 1: Kontras untuk mempertajam batas (misal alpha=1.2, beta=10)
    enhanced = cv.convertScaleAbs(crayfish_rgb, alpha=1.3, beta=20)

    # Langkah 2: Gamma correction untuk mencerahkan area redup tanpa oversaturating
    gamma_val = 0.8
    invGamma = 1.0 / gamma_val
    table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype('uint8')
    final_crayfish = cv.LUT(enhanced, table)

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1), plt.imshow(crayfish_rgb), plt.title('Before (Original)')
    plt.subplot(1, 2, 2), plt.imshow(final_crayfish), plt.title('After (Contrast + Gamma)')
    plt.show()


**Analisa Perbaikan `crayfish.jpg`:**
- **Metode yang dipilih:** Kombinasi *Linear Contrast* & *Gamma Correction*.
- **Alasan pemilihan:** Citra aslinya seringkali terlihat pudar, kekurangan rentang dinamis, dan berkabut. Menggunakan *contrast* akan membuat perbedaan antara warna lebih mencolok (melebarkan kurva histogram), sementara *gamma correction* digunakan untuk mengatur distribusi cahaya keseluruhan agar detail capit atau anatomi udang (crayfish) lebih terlihat jelas dari background yang redup.
- **Parameter Terbaik:** (Sangat bergantung pada citra aslinya), namun umumnya nilai kontras (alpha) 1.2 hingga 1.5, dan gamma sedikit di bawah 1 (misal 0.7 - 0.9) memberikan iluminasi yang pas.
